In [ ]:
from PIL import Image
from io import BytesIO
import requests
from src.db.mysql import DbConnection

conn = DbConnection()
rows = conn.select_all("""
SELECT app_id_int, icon FROM scraped_data.apps
WHERE app_id_int NOT IN (
SELECT app_id_int FROM scraped_data.apps_icons
)
and icon is not null
""")

cnt = 0
rows_cnt = len(rows)
size = 128, 128
for row in rows:
    app_id_int = row["app_id_int"]
    icon = row["icon"]

    cnt += 1
    if cnt % 100 == 0:
        print(f"{cnt} / {rows_cnt}")

    if "%CDN_HOST_MEDIA_SSL%" in icon:
        continue

    try:
        response = requests.get(icon)
        img = Image.open(BytesIO(response.content))
        img.thumbnail(size, Image.LANCZOS)
        output = BytesIO()
        img.save(output, format='PNG')
        data = output.getvalue()
        conn.query("delete from scraped_data.apps_icons where app_id_int = %s", [app_id_int])
        conn.query("insert into scraped_data.apps_icons (app_id_int, icon) VALUES (%s, %s)",
               [app_id_int, data])
        conn.commit()
    except Exception as e:
        print(f"Error: {app_id_int}")
        print(e)
        continue

conn.close()